# Datamine SMUHIS Process Example

This notebook demonstrates how to configure and run the **`smuhis`** process wrapper in `dmstudio`.

## Process Description

## SMUHIS

Create a histogram using the 'Shortcut' evaluation method.

The process creates a histogram file by evaluating a kriged model using the 'Shortcut' method and estimating the tonnage distribution over the range of cutoffs specified for the histogram intervals. The data in this file can then be used to create grade/tonnage plots.

A Selective Mining Unit (SMU) is the smallest block size that can be mined selectively. The dimensions of an SMU are defined using parameters **SMUXINC** , **SMUYINC** and **SMUZINC**.The output histogram file shows the grade and tonnage for a range of cut-offs for the specified size of SMU.

The process takes account of the distribution of SMUs within each model cell or subcell. This distribution may be either normal (1) or lognormal (2); it is specified by parameter **SMUDIST**.

Care should be taken when using a normal distribution for SMUs (**SMUDIST** =1). Under certain conditions the method can give an overestimation of grade when the block grade is low. This is due to the fact that the theoretical normal distribution can have values below zero, which is not possible in practice. This can lead to the situation where the metal above a cut-off is estimated to be greater than the total metal for the whole block. In such a case a warning message is displayed. If a normal variogram model is fitted (**LOG** =1) but the distribution of the SMUs is lognormal (**SMUDIST** =2) then the mean of the data values (**DATAMEAN**) must be specified.

**INFOEFF** is the information effect variance. It is a constant representing the final production data variance correction and is added to the kriging variance. For further information refer to Journel and Huibrechts.

A similar problem of overestimation can also arise when using a lognormal distribution for SMUs and the grade is low compared to the additive constant (**ADDCON**). In such a case the additive constant should be reduced.

The dispersion variance of an SMU within a model cell or subcell can be defined using one of three methods as controlled by parameter **DVMETHOD**.

* =1 define a single dispersion variance using parameter **DISVAR**. This variance will then be used as the variance of an SMU within every cell and subcell in the model. This option is therefore most appropriate if all cells in the model are the same size.

* =2 define the [variogram model](<../COMMON/filetype.md#VariogramMod>) and the dimensions of the **SMU** by parameter. A single dispersion variance is then calculated for the SMU within a parent cell, and this value is used for all cells and subcells. This is similar to 1 above, except that the dispersion variance is calculated by the process rather than being supplied by the user through parameter DISVAR.

* =3 define the [variogram model](<../COMMON/filetype.md#VariogramMod>) and the dimensions of the **SMU** by parameter. An individual dispersion variance is then calculated for the SMU within each cell and subcell. This is therefore more appropriate than options 1 or 2 if the model contains different cell sizes.

The calculation of the dispersion variance is one of the more time consuming parts of the process if **DVMETHOD** =3. This can be speeded up by making certain approximations as described below. This option is controlled by parameter **VARTYPE**.

* =1 the exact dimensions of the subcell are used, and so the dispersion variance is calculated for every subcell in the model.

* =2 each subcell is approximated to one of a discrete number of subcells. As each subcell is processed reference is made to a look up table to see whether the dispersion variance for that size subcell has already been calculated. If it exists then the value is used; if not then it is calculated and stored in the table. This gives a large speed improvement

Whichever option is used the dispersion variance of an SMU in a parent cell is calculated once and stored. Therefore the increase in speed under option 2 will be most apparent when the model includes a large number of subcells.

If **DVMETHOD** =3 and **VARTYPE** =2 then each subcell is approximated by one of a discrete number of sizes. This is controlled by parameters **XSTEP , YSTEP** and **ZSTEP**. Dispersion variances are stored for subcells whose dimensions are an integer multiple of the step sizes. The maximum possible number of increments in each dimension is 20, making a total of 8000 (20**3) discrete sizes.

The user should ensure that the step size in each dimension is greater than or equal to one twentieth of the dimension of the parent cell for the model.

## XSTEP >= XINC / 20

If this is not the case then the step value will be reset as one twentieth of the parent cell dimension.

An optional bench perimeter can be used to constrain the evaluation. The standard perimeter fields can be supplemented by a **PCODE** field. Perimeters are evaluated assuming mid-bench position and partial cell evaluation.

The evaluation can include recovered fractions. If **RECOVERY** =f (in the range 0-1) then:

* if **FRACTION** < (1-f), treat f as zero grade
* if **FRACTION** > f, take the whole block at the average grade.

The output file contains the following fields:

PVALUE  |  perimeter identifier. Only created if PERIM was defined.
---|---
PCODE  |  secondary perimeter classification field. Only produced if PERIM and PCODE were defined.
LOWER, MIDDLE, and UPPER  |  provide histogram bin limits with LOWER being regarded as the cut-off value.
TONNE, TONNE-% and AVGRADE  |  provide tonnage and grade data for material reporting in the interval with full recovery.
CTONNE, CTONNE-% and CGRADE  |  provide the cumulative tonnage and grade above the LOWER cut-off grade with full recovery. CMETAL reports the metal content ie CGRADE * TONNE.
RECOVERY  |  is the mining recovery parameter.
ARFTONNE and ARFGRADE  |  provide a summary of material below cut-off for those blocks where the recovered fraction is greater than RECOVERY. This material is mined as ore, but is below cut-off.
BRFTONNE and BRFGRADE  |  provide a summary of material above cut-off for those blocks where the recovered fraction is below (1-RECOVERY). This material cannot be taken as ore, but is still above cut-off.
TOTONNE and TOGRADE  |  report the total ore tonnes that would be mined for the RECOVERY specified. It is calculated as: TOTONNE = CTONNE + ARFTONNE - BRFTONNE

The first record in the output file has absent data or zero values for most fields. The exceptions are:

* **CTONNE** (|  the total tonnes in the model):
---|---

## CTONNE-%:  |  100

* **CGRADE** (|  the total average grade):

* **CMETAL** (|  the total metal content):

* **TOTONNE** (|  same as **CTONNE**):

* **TOGRADE** (|  same as **CGRADE**):

This record provides a summary of the total model without any cut-off. This data is not otherwise provided if the lowest cut-off specified (MINIMUM) is other than zero.

### Input Files:

* **in** (Block Model):
  Input model file.
  Required=Yes

* **vmodparm** (Variogram - Model):
  Variogram model parameter file. A variogram model is only required if **DVMETHOD** = 2
  or 3.
  Required=No

* **perim** (String):
  Optional bench perimeter file.
  Required=No

### Output Files:

* **out** (Undefined):
  Output file. This will contain one record for each histogram bin plus one record for
  the total model.
  Required=Yes

### Fields:

* **value** (Numeric : IN):
  A field in the model file which contains the kriged grade estimate.
  Default=Undefined
  Required=Yes

* **variance** (Numeric : IN):
  A field in the model file which contains the kriging variance (as created by
  **[ESTIMA](<estima.md>)** or [ESTIMATE Process](<estimate.md>))
  Default=Undefined
  Required=Yes

* **pcode** (Numeric : PERIM):
  Optional perimeter key field.
  Default=Undefined
  Required=No

### Parameters:

* **dvmethod**:
  Method for calculating dispersion variance (3) Options: 1: by parameter **DISVAR**.; 2:
  user defines the variogram model and the dimensions of the SMU by parameter. A single
  dispersion variance is calculated for the SMU within a parent cell, and this value is
  used for all cells and subcells.; 3: user defines the variogram model and the dimensions
  of the SMU by parameter. A dispersion variance is calculated for the SMU within each
  cell and subcell in the model.
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **vartype**:
  Method for calculating the variance of a sample in a cell (1). This only applies if
  **DVMETHOD** = 3. Options: 1: the exact dimensions of the cell are used; 2: the cell is
  approximated to one of a discrete number of cells. The values for these cells are stored
  to avoid the need for recalculation for the cell of the same size. This gives a large
  speed improvement
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **smudist**:
  Distribution of SMUs: (1) Options: 1: normal; 2: lognormal; 3: truncated normal
  Range=1, 3
  Values=1, 2,3
  Default=1
  Required=No

* **disvar**:
  Dispersion variance value. (0) This is only required if **DVMETHOD** =1.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **density**:
  Default density if no density field is defined in the model.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **addcon**:
  Additive constant for three parameter lognormal distribution of SMUs. (0). This only
  applies if **SMUDIST** =2.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **recovery**:
  Value for recovered fraction (1). If the recovered fraction is greater than this value
  then the complete block is considered above cut-off. If the recovered fraction is less
  than 1- **RECOVERY** then none of the block is considered to be above cut-off.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **binsize**:
  Bin width for histogram. (1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **minimum**:
  Lower bound of first bin. If SMUDIST =3, then MINIMUM must be set to zero. (0)
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **numbins**:
  Number of bins, maximum 200. (50)
  Range=Undefined
  Values=Undefined
  Default=50
  Required=No

* **vmodnum**:
  Variogram model reference number as defined by the **VREFNUM** field in the
  **VMODPARM** file. This only applies if **DVMETHOD** = 2 or 3.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **log**:
  Log/Normal variogram flag. Default(0). The variogram model, as defined by VGRAM , is
  Normal if LOG =0 or Lognormal if LOG =1.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **smuxinc**:
  X dimension of the Selective Mining Unit. This only applies if **DVMETHOD** = 2 or 3.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **smuyinc**:
  Y dimension of the Selective Mining Unit. This only applies if **DVMETHOD**
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **smuzinc**:
  Z dimension of the Selective Mining Unit. This only applies if **DVMETHOD**
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **xstep**:
  X step size for subcell approximation in variance calculations. This is only used if
  **DVMETHOD** =3 and **VARTYPE** =2. This must be less than the parent cell dimension in

## X. (1)

  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ystep**:
  Y step size for subcell approximation in variance calculations. This is only used if
  **DVMETHOD** =3 and **VARTYPE** =2. This must be less than the parent cell dimension in

## Y. (1)

  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **zstep**:
  Z step size for subcell approximation in variance calculations. This is only used if
  **DVMETHOD** =3 and **VARTYPE** =2. This must be less than the parent cell dimension in

## Z. (1)

  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ipoints**:
  Number of discretisation points in X dimension to simulate model cell (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

* **jpoints**:
  Number of discretisation points in Y dimension to simulate model cell (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

* **kpoints**:
  Number of discretisation points in Z dimension to simulate model cell (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

* **datamean**:
  Mean grade of the input data (* **VALUE**). This is compulsory if both a normal
  variogram model is selected (@**LOG** =0) and a lognormal distribution of SMUs is
  selected (@**SMUDIST** =3). The value is used in the calculation of the variance.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **infoeff**:
  The information effect variance. (0) This is subtracted from the variance. Refer to
  Journel and Huijbregts, Mining Geostatistics, pp 449-454 for details.
  Range=Undefined
  Values=Undefined
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('smuhis')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute smuhis
print("Running smuhis...")
dm_cmd.smuhis(
    in_i='t_assays',  # required
    vmodparm_i='t_vpar',  # required
    perim_i='optional',  # required
    out_o='t_smuhis_out',  # required
    value_f='optional',  # required
    variance_f='optional',  # required
    # pcode_f='optional',  # optional
    # dvmethod_p=3,  # optional
    # vartype_p=1,  # optional
    # smudist_p=1,  # optional
    # disvar_p=0,  # optional
    # density_p=1,  # optional
    # addcon_p=0,  # optional
    # recovery_p=1,  # optional
    # binsize_p=1,  # optional
    # minimum_p=0,  # optional
    # numbins_p=50,  # optional
    # vmodnum_p=1,  # optional
    # log_p=0,  # optional
    # smuxinc_p=1,  # optional
    # smuyinc_p=1,  # optional
    # smuzinc_p=1,  # optional
    # xstep_p=1,  # optional
    # ystep_p=1,  # optional
    # zstep_p=1,  # optional
    # ipoints_p=6,  # optional
    # jpoints_p=6,  # optional
    # kpoints_p=6,  # optional
    # datamean_p='optional',  # optional
    # infoeff_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("smuhis execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_smuhis_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")